# Proyecto Integrador — Sismos en Perú
**Curso:** Lenguaje de Programación II
**Tema:** Análisis de la actividad sísmica reciente en Perú usando la API pública del USGS

**Requisitos obligatorios cubiertos en este notebook:**
1. Programación Orientada a Objetos (3 clases: `ExtractorSismos`, `ProcesadorSismos`, `VisualizadorSismos`)
2. Programas en red (uso de `requests` para HTTP)
3. Expresiones regulares (extracción de patrón en el texto del lugar del sismo)
4. API con información real y actualizada (USGS Earthquake API)
5. Procesamiento con pandas (limpieza, transformación, análisis)
6. Visualización (3 gráficos distintos)


## 1. Configuración inicial
Importamos las librerías y definimos los parámetros generales del análisis.

In [2]:
# Configuración inicial e importación de librerías
import requests
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Parámetros generales del análisis
DIAS_A_CONSULTAR = 90
MAGNITUD_MINIMA = 2.5

print("Configuración inicial lista.")
print(f"Días a consultar: {DIAS_A_CONSULTAR} | Magnitud mínima: {MAGNITUD_MINIMA}")

Configuración inicial lista.
Días a consultar: 90 | Magnitud mínima: 2.5


## 2. Clase `ExtractorSismos` (Programas en red / API)

Esta clase se conecta a la **API pública del USGS** (`https://earthquake.usgs.gov`), que es
gratuita, no requiere API key y entrega datos reales y actualizados de sismos en formato GeoJSON.
Se filtra solo el territorio peruano usando un *bounding box* de coordenadas.
Aquí se cumple el requisito de **Programas en red** (uso de `requests` para HTTP GET).

In [3]:
# Clase ExtractorSismos (conexión de red)
class ExtractorSismos:
    """Clase encargada de la conexión de red y obtención de datos de sismos en Perú."""

    LAT_MIN, LAT_MAX = -18.5, -0.0
    LON_MIN, LON_MAX = -81.5, -68.5

    def __init__(self, dias_atras: int = 90, magnitud_minima: float = 2.5):
        self.url_base = "https://earthquake.usgs.gov/fdsnws/event/1/query"
        self.dias_atras = dias_atras
        self.magnitud_minima = magnitud_minima
        self.datos_brutos = None

    def conectar_y_descargar(self) -> dict:
        """Realiza una solicitud HTTP GET a la API del USGS y descarga los sismos en Perú."""
        fecha_fin = datetime.utcnow()
        fecha_inicio = fecha_fin - timedelta(days=self.dias_atras)

        parametros = {
            "format": "geojson",
            "starttime": fecha_inicio.strftime("%Y-%m-%d"),
            "endtime": fecha_fin.strftime("%Y-%m-%d"),
            "minlatitude": self.LAT_MIN,
            "maxlatitude": self.LAT_MAX,
            "minlongitude": self.LON_MIN,
            "maxlongitude": self.LON_MAX,
            "minmagnitude": self.magnitud_minima,
            "orderby": "time",
        }

        try:
            print(f"[INFO] Conectando a la red: {self.url_base}")
            respuesta = requests.get(self.url_base, params=parametros, timeout=15)

            if respuesta.status_code == 200:
                self.datos_brutos = respuesta.json()
                total = len(self.datos_brutos.get("features", []))
                print(f"[INFO] Datos descargados correctamente. Sismos encontrados: {total}")
                return self.datos_brutos
            else:
                print(f"[ERROR] Código de estado HTTP inesperado: {respuesta.status_code}")
                return {}
        except requests.exceptions.RequestException as e:
            print(f"[CRÍTICO] Fallo en la conexión de red: {e}")
            return {}

# Ejecutamos la descarga real
extractor = ExtractorSismos(dias_atras=DIAS_A_CONSULTAR, magnitud_minima=MAGNITUD_MINIMA)
datos_crudos = extractor.conectar_y_descargar()


/var/folders/t7/3fh6s2vx5ljfpd6zmnh68v1w0000gn/T/ipykernel_1808/1768344437.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  fecha_fin = datetime.utcnow()


[INFO] Conectando a la red: https://earthquake.usgs.gov/fdsnws/event/1/query
[INFO] Datos descargados correctamente. Sismos encontrados: 49


## 3. Expresiones regulares (extracción de patrón)

La API devuelve el lugar del sismo como texto libre, por ejemplo: "34km SE of Lima, Peru". Usamos regex para extraer la distancia en km, la dirección y la ciudad/región de referencia. Esto cumple el requisito obligatorio de Expresiones regulares.

In [ ]:
# APORTE INTEGRANTE 3: patron regex (se usara como metodo dentro de ProcesadorSismos)
PATRON_LUGAR = r"(\d+)km\s+([NSEW]{1,2})\s+of\s+([A-Za-zÀ-ÿ\s]+),\s*Peru"

# Prueba rapida del patron con un texto de ejemplo
coincidencia = re.match(PATRON_LUGAR, "34km SE of Lima, Peru")
print(coincidencia.groups())  # (distancia_km, direccion, ciudad)

## 4. Clase `ProcesadorSismos` (Pandas) 

Convertimos el GeoJSON de la API en un `DataFrame`, limpiamos datos nulos, calculamos una
columna nueva (`Nivel` de intensidad) y ordenamos por fecha. Esto cumple el requisito de
**Procesamiento con pandas** (limpieza, transformación y análisis básico).

In [8]:

class ProcesadorSismos:
    """Clase encargada de limpiar, validar y transformar los datos de sismos."""

    def __init__(self, datos_api: dict):
        self.datos = datos_api
        self.df_limpio = None

    def extraer_info_lugar(self, texto_lugar: str):
        """ usa regex para extraer (distancia_km, direccion, ciudad)
        del texto del lugar del sismo.
        """
        coincidencia = re.match(PATRON_LUGAR, texto_lugar.strip())
        if coincidencia:
            distancia_km = int(coincidencia.group(1))
            direccion = coincidencia.group(2)
            ciudad = coincidencia.group(3).strip()
            return distancia_km, direccion, ciudad
        else:
            return None, None, texto_lugar.strip()

    def transformar_a_dataframe(self) -> pd.DataFrame:
        """APORTE INTEGRANTE 4: transforma el GeoJSON de la API en un DataFrame limpio."""
        if not self.datos or "features" not in self.datos:
            print("[ERROR] No hay datos válidos para procesar.")
            return pd.DataFrame()

        registros = []
        for sismo in self.datos["features"]:
            props = sismo.get("properties", {})
            geometria = sismo.get("geometry", {})
            coords = geometria.get("coordinates", [None, None, None])

            lugar_texto = props.get("place", "")
            distancia_km, direccion, ciudad = self.extraer_info_lugar(lugar_texto)

            registros.append({
                "Fecha": pd.to_datetime(props.get("time"), unit="ms", errors="coerce"),
                "Magnitud": props.get("mag"),
                "Lugar": lugar_texto,
                "Ciudad_Referencia": ciudad,
                "Distancia_km": distancia_km,
                "Direccion": direccion,
                "Profundidad_km": coords[2],
                "Latitud": coords[1],
                "Longitud": coords[0],
            })

        df = pd.DataFrame(registros)
        df = df.dropna(subset=["Magnitud", "Profundidad_km"])

        df["Nivel"] = pd.cut(
            df["Magnitud"],
            bins=[0, 3.9, 4.9, 5.9, 10],
            labels=["Leve", "Moderado", "Fuerte", "Muy Fuerte"],
        )

        df = df.sort_values(by="Fecha", ascending=False).reset_index(drop=True)
        self.df_limpio = df
        return self.df_limpio

procesador = ProcesadorSismos(datos_crudos)
df_final = procesador.transformar_a_dataframe()
df_final.head(10)


NameError: name 'PATRON_LUGAR' is not defined

## 5. Clase VisualizadorSismos (Visualización)

Generamos 3 graficos distintos a partir del DataFrame ya procesado, cumpliendo el requisito obligatorio de Visualizacion.

In [ ]:
class VisualizadorSismos:
    """Clase encargada de generar reportes visuales sobre los sismos en el Perú."""

    def __init__(self, dataframe: pd.DataFrame):
        self.df = dataframe

    def grafico_barras_por_nivel(self):
        """Grafico 1: Cantidad de sismos registrados según su nivel de intensidad."""
        plt.figure(figsize=(8, 5))
        orden = ["Leve", "Moderado", "Fuerte", "Muy Fuerte"]
        sns.countplot(x="Nivel", data=self.df, order=orden, palette="Reds")
        plt.title("Grafico 1: Cantidad de Sismos por Nivel de Intensidad (Perú)")
        plt.xlabel("Nivel de Intensidad")
        plt.ylabel("Cantidad de Sismos")
        plt.grid(axis="y", linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.show()

    def grafico_lineas_magnitud_tiempo(self):
        """Gráfico 2: Evolución de la magnitud de los sismos a lo largo del tiempo."""
        df_ordenado = self.df.sort_values(by="Fecha")
        plt.figure(figsize=(10, 5))
        plt.plot(df_ordenado["Fecha"], df_ordenado["Magnitud"], marker="o",
                 linestyle="-", color="darkred", alpha=0.7)
        plt.title("Gráfico 2: Evolución de la Magnitud de Sismos en Perú")
        plt.xlabel("Fecha")
        plt.ylabel("Magnitud")
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.4)
        plt.tight_layout()
        plt.show()

    def grafico_caja_profundidad(self):
        """Grafico 3: Diagrama de caja (Boxplot) de la profundidad de los sismos."""
        plt.figure(figsize=(7, 5))
        sns.boxplot(y=self.df["Profundidad_km"], color="orange")
        plt.title("Grafico 3: Dispersión de la Profundidad de los Sismos (km)")
        plt.ylabel("Profundidad (km)")
        plt.tight_layout()
        plt.show()

visualizador = VisualizadorSismos(df_final)

## 6. Integración final y resultados

Resumen estadístico del análisis y conclusión general.

In [ ]:
# Resumen final
if not df_final.empty:
    print(f"Total de sismos analizados: {len(df_final)}")
    print(f"Magnitud promedio: {df_final['Magnitud'].mean():.2f}")
    print(f"Sismo más fuerte: {df_final['Magnitud'].max():.2f} "
          f"({df_final.loc[df_final['Magnitud'].idxmax(), 'Lugar']})")
    print(f"Profundidad promedio: {df_final['Profundidad_km'].mean():.1f} km")
else:
    print("No se pudo generar el resumen porque no llegaron datos de la API.")

### Conclusión 

Los datos confirman que Perú, al ubicarse en el **Cinturón de Fuego del Pacífico**, registra
actividad sísmica constante, mayormente de intensidad leve a moderada. Este análisis combinó
consumo de API en tiempo real, expresiones regulares, procesamiento con pandas y visualización
de datos, aplicando Programación Orientada a Objetos en todo el flujo